## OKAERTool and PyNavis initialization

Board: XEM6310 Spartan-6

In [2]:
import sys
import os
import time

# Add the parent directory to the path to import pyOKAERTool (only if the package is not installed)
# sys.path.insert(0, os.path.abspath('..'))
import pyOKAERTool as okt
from pyNAVIS import *
import os

# Define bitfile path
bitfile_path = '../bitfiles/CNAS_okaertool_XEM6310.bit'
# bitfile_path = None  # Set to None if no .bit file is to be used

# Validate the existence of the .bit file
if bitfile_path is None:
    None
elif not os.path.exists(bitfile_path):
    print(f"El archivo .bit no existe en la ruta especificada: {bitfile_path}")
    sys.exit(1)

# Create a new intance of the OkaerTool class and initialize it
okaer = okt.Okaertool(bit_file=bitfile_path)
okaer.init()

# Create a new instance of the PyNAVIS class
settings = MainSettings(num_channels=64, mono_stereo=1, on_off_both=1, address_size=4, ts_tick=0.01, bin_size=10000)

06/12/26 05:06:54 PM - INFO : Device product ID: 22, product name: XEM6310-LX150, USB speed: 2,
06/12/26 05:06:54 PM - INFO : USB 2.0 HighSpeed. USB block size set to 1 KB
06/12/26 05:06:54 PM - INFO : okaertool initialized as idle


## NAS configuration

In [3]:
import re
import os

config_file_path = '../CFBank_64_20_22000.vhd'

def _tok_to_int(tok):
    tok = tok.strip().rstrip(',').strip()
    if tok.lower().startswith('x"') and tok.endswith('"'):
        return int(tok[2:-1], 16)
    if tok.lower().startswith('0x'):
        return int(tok, 16)
    m = re.match(r'16#([0-9A-Fa-f]+)#', tok)
    if m:
        return int(m.group(1), 16)
    if tok.isdigit():
        return int(tok, 10)
    raise ValueError(f"Unrecognized token: {tok!r}")

def parse_cascade_vhd(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    text = open(path, 'r', encoding='utf-8', errors='ignore').read()

    # Find successive groups of the four parameters in the file order
    pattern = re.compile(
        r'FREQ_DIV\s*=>\s*(?P<f>[^,\n;]+)\s*,\s*'
        r'SPIKES_DIV_FB\s*=>\s*(?P<fb>[^,\n;]+)\s*,\s*'
        r'SPIKES_DIV_OUT\s*=>\s*(?P<out>[^,\n;]+)\s*,\s*'
        r'SPIKES_DIV_BPF\s*=>\s*(?P<bpf>[^,\n;]+)',
        re.IGNORECASE | re.DOTALL
    )

    values = []
    for m in pattern.finditer(text):
        f = _tok_to_int(m.group('f'))
        fb = _tok_to_int(m.group('fb'))
        out = _tok_to_int(m.group('out'))
        bpf = _tok_to_int(m.group('bpf'))
        values.extend([f, fb, out, bpf])

    return values

def reset_and_configure_okaer():
    #Reset the OkaerTool
    okaer.reset_board(mode='internal')

    # Configure the PDM2Spikes (left and right) for both NAS
    register_address = 0x0000
    okaer.logger.info("Configuring PDM2Spikes modules")
    # Left cochlea
    okaer.logger.info("Left cochlea")
    for value in PDM2Spikes_DEFAULT_parameter:
        okaer.set_config('port_a', register_address, value)
        # okaer.set_config('port_b', register_address, value)
        register_address += 1
    # Right cochlea
    okaer.logger.info("Right cochlea")
    for value in PDM2Spikes_DEFAULT_parameter:
        okaer.set_config('port_a', register_address, value)
        # okaer.set_config('port_b', register_address, value)
        register_address += 1

    register_address = 0x08
    okaer.logger.info("Configuring I2S2Spikes modules")
    # Configure I2S2Spikes modules for both NAS
    for value in I2S2Spikes_DEFAULT_parameter:
        okaer.set_config('port_a', register_address, value)
        # okaer.set_config('port_b', register_address, value)

    # Configure the filters for CASCADE NAS
    okaer.logger.info("Configuring filters for Cascade NAS")
    # Left cochlea
    register_address = 0x09
    okaer.logger.info("Left cochlea")
    for value in CASCADE_FILTER_DEFAULT_parameter:
        okaer.set_config('port_a', register_address, value)
        register_address += 1
        # # Config only 32 filters
        # if register_address >= 0x09 + 32*4:
        #     break
    # Right cochlea
    register_address = 0x010D
    okaer.logger.info("Right cochlea")
    for value in CASCADE_FILTER_DEFAULT_parameter:
        okaer.set_config('port_a', register_address, value)
        register_address += 1
        # # Config only 32 filters
        # if register_address >= 0x010D + 32*4:
        #     break

# Define default parameters for the filters
PDM2Spikes_DEFAULT_parameter = [0x0005, 0x0006, 0x734B, 0x39C8]
I2S2Spikes_DEFAULT_parameter = [0x000F]
CASCADE_FILTER_DEFAULT_parameter = parse_cascade_vhd(config_file_path)

# quick validation / pretty print
filters = len(CASCADE_FILTER_DEFAULT_parameter) // 4
print(f"Parsed {filters} filters ({len(CASCADE_FILTER_DEFAULT_parameter)} values).")
print("CASCADE_FILTER_DEFAULT_parameter = [")
for v in CASCADE_FILTER_DEFAULT_parameter:
    # print as hex literal (4 hex digits minimum)
    width = max(2, (v.bit_length() + 3) // 4)
    print(f"    0x{v:0{width}X},")
print("]")

#reset_and_configure_okaer()

Parsed 65 filters (260 values).
CASCADE_FILTER_DEFAULT_parameter = [
    0x04,
    0x7CB1,
    0x7CB1,
    0x2025,
    0x04,
    0x6F93,
    0x6F93,
    0x2025,
    0x02,
    0x77CE,
    0x77CE,
    0x2025,
    0x02,
    0x6B33,
    0x6B33,
    0x2025,
    0x03,
    0x7FE5,
    0x7FE5,
    0x2025,
    0x03,
    0x7271,
    0x7271,
    0x2025,
    0x03,
    0x6666,
    0x6666,
    0x2025,
    0x04,
    0x7289,
    0x7289,
    0x2025,
    0x02,
    0x7AFB,
    0x7AFB,
    0x2025,
    0x02,
    0x6E0B,
    0x6E0B,
    0x2025,
    0x02,
    0x6277,
    0x6277,
    0x2025,
    0x03,
    0x757A,
    0x757A,
    0x2025,
    0x03,
    0x691E,
    0x691E,
    0x2025,
    0x04,
    0x7593,
    0x7593,
    0x2025,
    0x02,
    0x7E3F,
    0x7E3F,
    0x2025,
    0x02,
    0x70F7,
    0x70F7,
    0x2025,
    0x02,
    0x6514,
    0x6514,
    0x2025,
    0x03,
    0x7898,
    0x7898,
    0x2025,
    0x03,
    0x6BE8,
    0x6BE8,
    0x2025,
    0x04,
    0x78B1,
    0x78B1,
    0x2025,
    0x04,
 

## Experiment

### Audio functions

In [4]:
import sounddevice as sd
import soundfile as sf
import tkinter as tk
from tkinter import filedialog

def list_output_devices():
    """Prints all available audio output devices and their IDs."""
    print(sd.query_devices())

def select_wav_folder():
    root = tk.Tk()
    root.withdraw()  # Hide the main window
    folder_path = filedialog.askdirectory(title="Select a folder containing WAV files")
    return folder_path


def collect_wav_files(folder_path):
    wav_files = []
    for dirpath, _, filenames in os.walk(folder_path):
        for filename in sorted(filenames):
            if filename.lower().endswith('.wav'):
                wav_files.append(os.path.join(dirpath, filename))
    return wav_files


def get_wav_header_info(file_path):
    """Extracts metadata (features) from the WAV header."""
    with sf.SoundFile(file_path) as f:
        info = {
            "samplerate": f.samplerate,
            "channels": f.channels,
            "subtype": f.subtype,      # Bit depth (e.g., PCM_16)
            "format": f.format,        # File format (WAV, FLAC, etc.)
            "frames": f.frames,        # Total number of audio samples
            "duration_sec": f.frames / f.samplerate
        }
    return info

def play_audio_on_device(data, fs, device_id, block=True):
    """
    Plays audio data through a specific output interface.
    :param data: The audio data to be played
    :param fs: The sample rate of the audio data
    :param device_id: The ID of the device (from list_output_devices)
    """
    try:
        sd.default.device = device_id
        print(f"Playing on device {device_id}...")
        sd.play(data, fs)
        if block:
            sd.wait()
    except Exception as e:
        print(f"Error: {e}")

output_device = None
if output_device is None:
    list_output_devices()
    output_device = int(input("Please set the output_device variable to the ID of your desired output device: "))


   0 Asignador de sonido Microsoft - Input, MME (2 in, 0 out)
>  1 Micrófono (Realtek(R) Audio), MME (2 in, 0 out)
   2 Asignador de sonido Microsoft - Output, MME (0 in, 2 out)
<  3 Realtek HD Audio 2nd output (Re, MME (0 in, 2 out)
   4 PLG2773 (NVIDIA High Definition, MME (0 in, 2 out)
   5 Altavoces (Realtek(R) Audio), MME (0 in, 2 out)
   6 Controlador primario de captura de sonido, Windows DirectSound (2 in, 0 out)
   7 Micrófono (Realtek(R) Audio), Windows DirectSound (2 in, 0 out)
   8 Controlador primario de sonido, Windows DirectSound (0 in, 2 out)
   9 Realtek HD Audio 2nd output (Realtek(R) Audio), Windows DirectSound (0 in, 2 out)
  10 PLG2773 (NVIDIA High Definition Audio), Windows DirectSound (0 in, 2 out)
  11 Altavoces (Realtek(R) Audio), Windows DirectSound (0 in, 2 out)
  12 PLG2773 (NVIDIA High Definition Audio), Windows WASAPI (0 in, 2 out)
  13 Realtek HD Audio 2nd output (Realtek(R) Audio), Windows WASAPI (0 in, 2 out)
  14 Altavoces (Realtek(R) Audio), Windows W

### Playing audio and monitoring spikes

In [ ]:
import matplotlib.pyplot as plt
import AERzip
import threading
import numpy as np

# Monitor the inputs
INPUTS = ['port_a']  # Monitor only port_a where the CNAS outputs are sent. port_b is not used in this configuration
USB_TRANSFER_LENGTH = 64 * 1024

# Set USB transfer length and number of buffers
okaer.USB_TRANSFER_LENGTH = USB_TRANSFER_LENGTH

# Reset the okaerTool board before monitoring to ensure a clean state
okaer.reset_board(mode='internal')

# Set logger to ERROR level to reduce verbosity during processing
okaer.logger.setLevel(okt.logging.ERROR)

# Set up base directories
base_plot = '../Plots'
base_comp = '../Compressed files'

wav_folder_path = select_wav_folder()
if wav_folder_path:
    wav_files = collect_wav_files(wav_folder_path)
    if not wav_files:
        print(f"No WAV files found in the selected folder: {wav_folder_path}")
    else:
        print(f"Found {len(wav_files)} WAV files in {wav_folder_path}")

        # Create subfolder names based on the selected folder
        folder_name = os.path.basename(wav_folder_path)
        target_plot = os.path.join(base_plot, folder_name)
        target_comp = os.path.join(base_comp, folder_name)
        os.makedirs(target_plot, exist_ok=True)
        os.makedirs(target_comp, exist_ok=True)

        for wav_file in wav_files:
            root = os.path.dirname(wav_file)
            file = os.path.basename(wav_file)
            file_base = os.path.splitext(file)[0]

            # Check if output files already exist and skip processing if they do
            plot_path = os.path.join(target_plot, file_base + '_sonogram.png')
            aer_path = os.path.join(target_comp, file_base + '_spikes.aedat')
            if os.path.exists(plot_path) and os.path.exists(aer_path):
                print(f"Skipping {file}: Output files already exist")
                continue

            print(f"Processing audio file: {file} (from {root})")

            wav_info = get_wav_header_info(wav_file)
            duration = wav_info['duration_sec']  # Audio length plus small buffer

            spikes_result = {}
            monitor_started = threading.Event()
            monitor_done = threading.Event()

            def monitor_spikes():
                monitor_started.set()  # Signal that monitor is beginning
                try:
                    spikes_result['spikes'] = okaer.monitor(inputs=INPUTS, duration=duration)
                finally:
                    monitor_done.set()  # Signal that monitor is complete

            monitor_thread = threading.Thread(target=monitor_spikes)

            okaer.logger.info("Monitoring for a duration of %.2f seconds", duration)
            reset_and_configure_okaer()  # Ensure the board is reset and configured before monitoring
            data, fs = sf.read(wav_file)  # Preload audio so playback starts immediately

            monitor_thread.start()

            # Wait for monitoring to start (strict synchronization)
            if not monitor_started.wait(timeout=2.0):
                okaer.logger.error("Monitor did not start in time for %s. Skipping.", file)
                continue

            play_audio_on_device(data, fs, output_device, block=False)
            print("Playing audio...")

            # Wait for monitoring to complete
            monitor_done.wait()
            
            spikes = spikes_result.get('spikes')
            if not spikes:
                okaer.logger.error("No spikes were recorded for %s. Skipping.", file)
                continue

            monitored = spikes[0]
            addr_len = len(monitored.addresses)
            ts_len = len(monitored.timestamps)
            if addr_len != ts_len:
                okaer.logger.error("Mismatch: %d addresses vs %d timestamps!", addr_len, ts_len)
                continue

            if monitored.get_num_spikes() == 0:
                okaer.logger.warning("Input %s: No spikes recorded", INPUTS[0])
                continue

            timestamps = np.asarray(monitored.timestamps)
            addresses = np.asarray(monitored.addresses)

            okaer.logger.info("Spike data OK: %d events.", addr_len)
            okaer.logger.info("Input %d: %d spikes", 0, monitored.get_num_spikes())

            min_ts = timestamps.min()
            max_ts = timestamps.max()
            min_addr = addresses.min()
            max_addr = addresses.max()
            ts_tick_us = 0.01  # Each tick = 10ns = 0.01 microseconds

            okaer.logger.info("--- Input %s ---", INPUTS[0])
            okaer.logger.info("Total spikes: %d", len(timestamps))
            okaer.logger.info("Timestamp range (ticks): %d - %d", min_ts, max_ts)
            okaer.logger.info("Timestamp range (us): %.2f - %.2f", min_ts * ts_tick_us, max_ts * ts_tick_us)
            okaer.logger.info("Duration (ms): %.2f", (max_ts - min_ts) * ts_tick_us / 1000)
            okaer.logger.info("Address range: %d - %d", min_addr, max_addr)

            if not np.all(timestamps[:-1] <= timestamps[1:]):
                okaer.logger.error("Timestamps are NOT in ascending order!")
                bad_idx = np.where(timestamps[:-1] > timestamps[1:])[0][0]
                okaer.logger.error("First violation at index %d: %d > %d", bad_idx, timestamps[bad_idx], timestamps[bad_idx + 1])
            else:
                okaer.logger.info("Timestamps are in ascending order")

            if np.any(timestamps < 0):
                okaer.logger.error("Found %d negative timestamps!", np.sum(timestamps < 0))
            else:
                okaer.logger.info("No negative timestamps")

            if len(timestamps) > 1:
                deltas = np.diff(timestamps)
                mean_delta_ns = np.mean(deltas) * 10
                median_delta_ns = np.median(deltas) * 10
                max_delta_ns = np.max(deltas) * 10
                okaer.logger.info(
                    "Timestamp deltas (ns): mean=%.1f, median=%.1f, max=%.1f",
                    mean_delta_ns,
                    median_delta_ns,
                    max_delta_ns,
                )
                if max_addr - min_addr > 200:
                    okaer.logger.info("Expected delta for sequential scan: ~240ns (24 ticks @ 10ns)")

            duration_s = (max_ts - min_ts) * ts_tick_us / 1e6
            if duration_s > 0:
                event_rate = len(timestamps) / duration_s
                okaer.logger.info("Event rate: %.0f spikes/sec", event_rate)
                if event_rate > 10_000_000:
                    okaer.logger.warning("Event rate seems very high: %.0f spikes/sec", event_rate)
                elif event_rate < 100:
                    okaer.logger.warning("Event rate seems very low: %.0f spikes/sec", event_rate)
                else:
                    okaer.logger.info("Event rate within reasonable range")

            unique_addrs = np.unique(addresses)
            okaer.logger.info("Unique addresses: %d", len(unique_addrs))
            okaer.logger.info("Address range: %d to %d", min_addr, max_addr)

            if len(unique_addrs) > 10:
                expected_sequential = np.arange(min_addr, max_addr + 1)
                if np.array_equal(unique_addrs, expected_sequential):
                    okaer.logger.info("Addresses are sequential (as expected after reset)")
                else:
                    missing = np.setdiff1d(expected_sequential, unique_addrs)
                    if missing.size:
                        okaer.logger.info("Some addresses missing: %s...", missing[:10].tolist())

            addr_counts = np.bincount(addresses.astype(np.int64, copy=False))
            top_10_indices = np.argsort(addr_counts)[-10:][::-1]
            okaer.logger.info("Top 10 addresses by count:")
            for addr in top_10_indices:
                count = addr_counts[addr]
                if count > 0:
                    okaer.logger.info("  Address %d: %d events", addr, count)

            okaer.logger.info("Plotting the sonogram for input %s", INPUTS[0])
            Plots.sonogram(SpikesFile(addresses=monitored.addresses, timestamps=monitored.timestamps), settings)
            plt.savefig(plot_path)
            plt.close()

            # Save event file inside the folder named after the selected folder
            AERzip.saveCompressedFile(addresses, timestamps, aer_path, overwrite=True)

        print(f"Finished processing {len(wav_files)} WAV files.")
else:
    print("No folder was selected.")


06/12/26 05:07:03 PM - INFO : Board reset in mode: internal


Found 460 WAV files in C:/Users/alvco/Desktop/Manchester2026/Dataset/1
Processing audio file: r_0.wav (from C:/Users/alvco/Desktop/Manchester2026/Dataset/1)
Playing on device 3...
Playing audio...
Processing audio file: r_1.wav (from C:/Users/alvco/Desktop/Manchester2026/Dataset/1)
Playing on device 3...
Playing audio...
Processing audio file: r_10.wav (from C:/Users/alvco/Desktop/Manchester2026/Dataset/1)
Playing on device 3...
Playing audio...
Processing audio file: r_100.wav (from C:/Users/alvco/Desktop/Manchester2026/Dataset/1)
Playing on device 3...
Playing audio...
Processing audio file: r_101.wav (from C:/Users/alvco/Desktop/Manchester2026/Dataset/1)
Playing on device 3...
Playing audio...
Processing audio file: r_102.wav (from C:/Users/alvco/Desktop/Manchester2026/Dataset/1)
Playing on device 3...
Playing audio...
